<a href="https://colab.research.google.com/github/sumairdawani/Bus-118-/blob/main/react_code_generation_expenses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2 Code Generation with ReACT Prompting

**Tools used:** Google Colab with Python 3, ChatGPT for prompt drafting, and the Python standard library. The notebook executes the generated code locally so the run and observation steps are visible without an API key.

**Goal:** Use a ReACT-style cycle to plan a solution, generate code, run it, observe an error, repair the code, and verify the final output.


## Full ReACT prompt

~~~text
You are a Python coding agent. Solve the task with a visible ReACT cycle and keep the reasoning concise.

Task: Generate a function named summarize_expenses(rows). Each row has a category and amount. Return a dictionary with category totals, a rounded grand total, and invalid_rows. Numeric strings such as "20.00" are valid. Negative amounts, missing fields, and non-numeric amounts must be reported in invalid_rows instead of crashing the program.

Stage 1 Reason and plan: describe the input assumptions, output schema, validation rules, and one edge case.
Stage 2 Act: generate Python code using only the standard library. Keep the function deterministic and avoid network calls.
Stage 3 Run: execute the generated code on the supplied test rows.
Stage 4 Observe: report the output or the exact error. Check whether the edge-case rules were satisfied.
Stage 5 Fix: if the run fails or a requirement is missing, revise the code and run it again.
Stage 6 Final check: print the final code result and state why it satisfies the requirements.
~~~

The first draft intentionally omits type conversion and validation. That creates a visible observation and repair cycle instead of hiding the iteration.


In [ ]:
import json

transactions = [
    {"category": "food", "amount": 12.50},
    {"category": "travel", "amount": 30.00},
    {"category": "food", "amount": 8.75},
    {"category": "other", "amount": -5.00},
    {"category": "travel", "amount": "20.00"},
]

plan = [
    "Accept rows with category and amount fields.",
    "Convert numeric strings to floats and reject negative or malformed amounts.",
    "Return totals, grand_total, and invalid_rows with stable rounding.",
    "Run a first draft, observe the failure, then repair and rerun.",
]

draft_code = """
def summarize_expenses(rows):
    totals = {}
    for row in rows:
        category = row["category"].strip().title()
        totals[category] = totals.get(category, 0) + row["amount"]
    return totals
"""

print("REASON / PLAN")
for item in plan:
    print("- " + item)
print("\nACT: generated first draft")
print(draft_code.strip())

first_namespace = {}
try:
    exec(draft_code, first_namespace)
    first_result = first_namespace["summarize_expenses"](transactions)
    print("\nOBSERVE: first draft returned", first_result)
except Exception as error:
    print("\nOBSERVE: first draft failed as expected")
    print(type(error).__name__ + ": " + str(error))

fixed_code = """
def summarize_expenses(rows):
    totals = {}
    invalid_rows = []
    for index, row in enumerate(rows):
        try:
            category = str(row["category"]).strip().title()
            if not category:
                raise ValueError("empty category")
            amount = float(row["amount"])
            if amount < 0:
                raise ValueError("negative amount")
        except (KeyError, TypeError, ValueError) as error:
            invalid_rows.append({"row": index, "reason": str(error)})
            continue
        totals[category] = round(totals.get(category, 0.0) + amount, 2)
    return {
        "totals": totals,
        "grand_total": round(sum(totals.values()), 2),
        "invalid_rows": invalid_rows,
    }
"""

print("\nFIX: revised code adds conversion, validation, and invalid_rows")
print(fixed_code.strip())
final_namespace = {}
exec(fixed_code, final_namespace)
final_result = final_namespace["summarize_expenses"](transactions)
assert final_result["totals"] == {"Food": 21.25, "Travel": 50.0}
assert final_result["grand_total"] == 71.25
assert final_result["invalid_rows"] == [{"row": 3, "reason": "negative amount"}]
print("\nOBSERVE: repaired code ran successfully")
print(json.dumps(final_result, indent=2))
print("\nFINAL CHECK: totals, string conversion, and negative-value handling passed.")


REASON / PLAN
- Accept rows with category and amount fields.
- Convert numeric strings to floats and reject negative or malformed amounts.
- Return totals, grand_total, and invalid_rows with stable rounding.
- Run a first draft, observe the failure, then repair and rerun.

ACT: generated first draft
def summarize_expenses(rows):
    totals = {}
    for row in rows:
        category = row["category"].strip().title()
        totals[category] = totals.get(category, 0) + row["amount"]
    return totals

OBSERVE: first draft failed as expected
TypeError: unsupported operand type(s) for +: 'float' and 'str'

FIX: revised code adds conversion, validation, and invalid_rows
def summarize_expenses(rows):
    totals = {}
    invalid_rows = []
    for index, row in enumerate(rows):
        try:
            category = str(row["category"]).strip().title()
            if not category:
                raise ValueError("empty category")
            amount = float(row["amount"])
            if amount < 